# 딴길 — 데이터 파이프라인

**실행 순서:** 셀을 위에서 아래로 순서대로 실행하세요.

**총 소요 시간:** 약 30분
- KOPIS 수집: ~5분
- LLM 프로파일링: ~10분 (300건)
- 임베딩 생성: ~3분 (GPU)
- 통합 JSON: ~1분

**사전 준비:**
1. 런타임 → 런타임 유형 변경 → **T4 GPU** 선택
2. 아래 셀(0. API 키)에 키 직접 입력

## 0. API 키 설정

In [ ]:
KOPIS_KEY     = "여기에_KOPIS_API_키_입력"
ANTHROPIC_KEY = "여기에_ANTHROPIC_API_키_입력"

assert KOPIS_KEY     != "여기에_KOPIS_API_키_입력",     "KOPIS_KEY를 입력해주세요."
assert ANTHROPIC_KEY != "여기에_ANTHROPIC_API_키_입력", "ANTHROPIC_KEY를 입력해주세요."

print("✅ API 키 설정 완료")

## 1. 라이브러리 설치

In [ ]:
!pip install anthropic sentence-transformers tqdm -q
print("✅ 설치 완료")

## 2. KOPIS 데이터 수집

In [ ]:
import requests
import xml.etree.ElementTree as ET
import json
import time
from datetime import datetime, timedelta
from tqdm import tqdm
from google.colab import files

BASE_URL = "http://kopis.or.kr/openApi/restful"

GENRE_CODES = {
    "뮤지컬":     "GGGA",
    "연극":       "AAAA",
    "음악":       "CCCA",
    "무용":       "BBBC",
    "국악":       "CCCC",
    "서커스/마술": "EEEB",
}

TARGETS = {
    "뮤지컬":     90,
    "연극":       75,
    "음악":       60,
    "무용":       30,
    "국악":       30,
    "서커스/마술": 15,
}

end_date   = datetime.today()
start_date = end_date - timedelta(days=180)
STDATE = start_date.strftime("%Y%m%d")
EDDATE = end_date.strftime("%Y%m%d")

def safe_xml_parse(content):
    try:
        return ET.fromstring(content)
    except ET.ParseError:
        return None

def fetch_list(genre_code, rows=100, page=1):
    params = {
        "service": KOPIS_KEY,
        "stdate": STDATE,
        "eddate": EDDATE,
        "cpage": page,
        "rows": rows,
        "shcate": genre_code,
    }
    try:
        resp = requests.get(f"{BASE_URL}/pblprfr", params=params, timeout=10)
        root = safe_xml_parse(resp.content)
        return root.findall(".//db") if root is not None else []
    except Exception:
        return []

def fetch_detail(perf_id):
    try:
        resp = requests.get(
            f"{BASE_URL}/pblprfr/{perf_id}",
            params={"service": KOPIS_KEY},
            timeout=10
        )
        root = safe_xml_parse(resp.content)
        if root is None:
            return {}
        db = root.find(".//db")
        if db is None:
            return {}
        return {
            "description": db.findtext("sty", ""),
            "cast":        db.findtext("prfcast", ""),
            "price":       db.findtext("pcseguidance", ""),
            "age_rating":  db.findtext("prfage", ""),
            "runtime":     db.findtext("prfruntime", ""),
            "venue_id":    db.findtext("mt10id", ""),
        }
    except Exception:
        return {}

def fetch_venue(venue_id):
    if not venue_id:
        return {}
    try:
        resp = requests.get(
            f"{BASE_URL}/prfplc/{venue_id}",
            params={"service": KOPIS_KEY},
            timeout=10
        )
        root = safe_xml_parse(resp.content)
        if root is None:
            return {}
        db = root.find(".//db")
        if db is None:
            return {}
        seats_raw = db.findtext("seatscale", "0").replace(",", "")
        return {
            "venue_seats":   int(seats_raw) if seats_raw.isdigit() else 0,
            "venue_address": db.findtext("adres", ""),
            "venue_lat":     db.findtext("la", ""),
            "venue_lng":     db.findtext("lo", ""),
        }
    except Exception:
        return {}

all_performances = []
seen_ids = set()
total_target = sum(TARGETS.values())

overall_bar = tqdm(total=total_target, desc="전체 진행", unit="건")

for genre_name, genre_code in GENRE_CODES.items():
    target    = TARGETS.get(genre_name, 20)
    collected = 0
    page      = 1

    genre_bar = tqdm(total=target, desc=f"  {genre_name}", unit="건", leave=False)

    while collected < target:
        items = fetch_list(genre_code, rows=min(100, target - collected + 10), page=page)
        if not items:
            break

        for item in items:
            if collected >= target:
                break
            perf_id = item.findtext("mt20id", "")
            if not perf_id or perf_id in seen_ids:
                continue

            title = item.findtext("prfnm", "")
            perf = {
                "id":         perf_id,
                "title":      title,
                "genre":      genre_name,
                "start_date": item.findtext("prfpdfrom", ""),
                "end_date":   item.findtext("prfpdto", ""),
                "venue_name": item.findtext("fcltynm", ""),
                "poster_url": item.findtext("poster", ""),
                "region":     item.findtext("area", ""),
                "state":      item.findtext("prfstate", ""),
            }

            detail = fetch_detail(perf_id)
            perf.update(detail)
            venue = fetch_venue(detail.get("venue_id", ""))
            perf.update(venue)

            all_performances.append(perf)
            seen_ids.add(perf_id)
            collected += 1

            genre_bar.set_postfix_str(title[:20])
            genre_bar.update(1)
            overall_bar.update(1)
            time.sleep(0.3)

        page += 1
        if page > 10:
            break

    genre_bar.close()

overall_bar.close()

with open("kopis_raw.json", "w", encoding="utf-8") as f:
    json.dump(all_performances, f, ensure_ascii=False, indent=2)

print(f"✅ {len(all_performances)}건 수집 완료 → 다운로드 시작")
files.download('kopis_raw.json')

## 3. LLM 욕망 프로파일링 (Claude Haiku)

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=ANTHROPIC_KEY)

PROFILING_SYSTEM = """당신은 공연예술 비평가이자 관객 심리 분석가입니다.
주어진 공연 정보를 분석하여 아래 JSON 형식으로 응답하세요.
JSON 외 다른 텍스트는 포함하지 마세요.

분석 원칙:
1. "장르"가 아니라 "이 공연이 관객에게 주는 감각 경험"을 서술하세요.
2. 이 공연과 감각적으로 연결되는 대중 콘텐츠를 2~3개 떠올리고,
   "같은 욕망이 다른 형식으로 충족되는 관계"를 설명하세요.
3. 대중성 스펙트럼 위치를 1(매우 대중적)~10(매우 실험적)으로 판단하세요.

{
  "desire_profile": {
    "core_desire": "이 공연이 만족시키는 핵심 욕망 (1문장)",
    "sensory_experience": "관객이 느끼는 감각적 경험 (1문장)",
    "emotional_payoff": "관람 후 남는 감정 (1문장)"
  },
  "mainstream_bridges": [
    {
      "mainstream_name": "연결 가능한 대중 콘텐츠명",
      "shared_desire": "공유하는 욕망 (1문장)",
      "key_difference": "형식적 차이 (1문장)"
    }
  ],
  "sensory_tags": ["감각 태그 5~8개"],
  "emotion_tags": ["감정 태그 2~4개"],
  "spectrum_position": 1,
  "entry_level": "쉬움/보통/어려움",
  "solo_friendly": true,
  "reason_hidden": "기존 플랫폼에서 덜 노출된 이유 (1문장)"
}"""

def profile_performance(perf):
    desc = perf.get('description', '').strip()
    if not desc or len(desc) < 10:
        desc = f"{perf['genre']} 공연"

    user_msg = f"""공연명: {perf['title']}
장르: {perf['genre']}
소개: {desc[:500]}
공연장: {perf.get('venue_name', '')} ({perf.get('venue_seats', '정보없음')}석)
지역: {perf.get('region', '')}
가격: {perf.get('price', '정보없음')}"""

    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=2000,
        system=PROFILING_SYSTEM,
        messages=[{"role": "user", "content": user_msg}]
    )

    if not response.content:
        raise ValueError(f"빈 응답 — stop_reason: {response.stop_reason}")

    raw = response.content[0].text.strip()

    if not raw:
        raise ValueError(f"빈 텍스트 — stop_reason: {response.stop_reason}")

    if raw.startswith("```"):
        lines = raw.splitlines()
        raw = "\n".join(lines[1:-1] if lines[-1].strip() == "```" else lines[1:])

    return json.loads(raw)

def profile_with_retry(perf, retries=2):
    for attempt in range(retries + 1):
        try:
            return profile_performance(perf)
        except Exception as e:
            tqdm.write(f"     [{attempt+1}/{retries+1}] 실패: {e}")
            if attempt < retries:
                time.sleep(2 ** attempt)
            else:
                raise e

with open("kopis_raw.json", "r", encoding="utf-8") as f:
    performances = json.load(f)

profiles = {}
failed   = []
SAVE_INTERVAL = 50

for i, perf in enumerate(tqdm(performances, desc="프로파일링", unit="건")):
    tqdm.write(f"  → {perf['title'][:30]}")
    try:
        profile = profile_with_retry(perf)
        profiles[perf['id']] = profile
    except Exception as e:
        failed.append(perf['id'])
        profiles[perf['id']] = None
        tqdm.write(f"     ✗ 최종 실패: {e}")
    time.sleep(0.5)

    if (i + 1) % SAVE_INTERVAL == 0:
        with open("profiles.json", "w", encoding="utf-8") as f:
            json.dump(profiles, f, ensure_ascii=False, indent=2)
        tqdm.write(f"\n💾 중간 저장 ({i+1}건) → 다운로드")
        files.download('profiles.json')

with open("profiles.json", "w", encoding="utf-8") as f:
    json.dump(profiles, f, ensure_ascii=False, indent=2)

print(f"\n✅ 프로파일링 완료: {len(performances) - len(failed)}/{len(performances)}건 → 다운로드 시작")
if failed:
    print(f"   실패: {len(failed)}건")
files.download('profiles.json')

## 4. 임베딩 생성 (GPU 활용)

In [ ]:
import torch
from sentence_transformers import SentenceTransformer
import numpy as np

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ 디바이스: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")

embed_model = SentenceTransformer('jhgan/ko-sroberta-multitask', device=device)
print("✅ 임베딩 모델 로드 완료 (768차원)")

with open("kopis_raw.json", "r", encoding="utf-8") as f:
    performances = json.load(f)
with open("profiles.json", "r", encoding="utf-8") as f:
    profiles = json.load(f)

def build_embedding_text(perf, profile):
    parts = [
        f"공연: {perf['title']}",
        f"장르: {perf['genre']}",
        f"소개: {perf.get('description', '')[:300]}",
    ]
    if profile:
        dp = profile.get('desire_profile', {})
        parts.extend([
            f"핵심 욕망: {dp.get('core_desire', '')}",
            f"감각 경험: {dp.get('sensory_experience', '')}",
            f"감정: {dp.get('emotional_payoff', '')}",
        ])
        for bridge in profile.get('mainstream_bridges', []):
            parts.append(
                f"대중 연결: {bridge.get('mainstream_name', '')} — "
                f"{bridge.get('shared_desire', '')}"
            )
        tags = profile.get('sensory_tags', [])
        if tags:
            parts.append(f"감각: {', '.join(tags)}")
    return " | ".join(parts)

texts = []
ids   = []
for perf in performances:
    profile = profiles.get(perf['id'])
    texts.append(build_embedding_text(perf, profile))
    ids.append(perf['id'])

print(f"임베딩 생성 중... ({len(texts)}건)")
embeddings = embed_model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True
)

embedding_map = {pid: emb.tolist() for pid, emb in zip(ids, embeddings)}

with open("embeddings.json", "w", encoding="utf-8") as f:
    json.dump(embedding_map, f)

print(f"✅ 임베딩 생성 완료: {len(embedding_map)}건, 차원: {len(next(iter(embedding_map.values())))} → 다운로드 시작")
files.download('embeddings.json')

## 5. 인기도 산출 + 통합 JSON 생성

In [ ]:
def calculate_popularity(venue_seats):
    seats = int(venue_seats) if str(venue_seats).isdigit() else 0
    if seats >= 1000:
        return min(90 + (seats - 1000) / 100, 100)
    elif seats >= 500:
        return 70 + (seats - 500) / 25
    elif seats >= 200:
        return 40 + (seats - 200) / 10
    elif seats >= 50:
        return 10 + (seats - 50) / 5
    else:
        return max(seats / 5, 1)

with open("kopis_raw.json",  "r", encoding="utf-8") as f:
    performances = json.load(f)
with open("profiles.json",   "r", encoding="utf-8") as f:
    profiles = json.load(f)
with open("embeddings.json", "r", encoding="utf-8") as f:
    embeddings = json.load(f)

contents = {}

for perf in tqdm(performances, desc="통합 JSON 생성", unit="건"):
    pid       = perf['id']
    profile   = profiles.get(pid)
    embedding = embeddings.get(pid)

    if not embedding:
        continue

    seats = perf.get('venue_seats', 0)
    contents[pid] = {
        "id":            pid,
        "title":         perf.get('title', ''),
        "genre":         perf.get('genre', ''),
        "description":   perf.get('description', ''),
        "venue_name":    perf.get('venue_name', ''),
        "venue_seats":   seats,
        "venue_address": perf.get('venue_address', ''),
        "venue_lat":     perf.get('venue_lat', ''),
        "venue_lng":     perf.get('venue_lng', ''),
        "region":        perf.get('region', ''),
        "start_date":    perf.get('start_date', ''),
        "end_date":      perf.get('end_date', ''),
        "price":         perf.get('price', ''),
        "age_rating":    perf.get('age_rating', ''),
        "poster_url":    perf.get('poster_url', ''),
        "state":         perf.get('state', ''),
        "popularity":    round(calculate_popularity(seats), 1),
        "profile":       profile,
        "embedding":     embedding,
    }

with open("contents.json", "w", encoding="utf-8") as f:
    json.dump(contents, f, ensure_ascii=False, indent=2)

from collections import Counter
print(f"✅ contents.json 생성 완료: {len(contents)}건")
print("\n장르 분포:")
for genre, cnt in Counter(v['genre'] for v in contents.values()).most_common():
    print(f"  {genre}: {cnt}건")
print("\n지역 분포 (상위 10):")
for region, cnt in Counter(v['region'] for v in contents.values()).most_common(10):
    print(f"  {region}: {cnt}건")

print("\n→ 다운로드 시작")
files.download('contents.json')

In [ ]:
from google.colab import files

files.download('contents.json')   # Next.js에서 사용하는 핵심 파일
files.download('kopis_raw.json')  # 백업용
print("✅ 다운로드 완료 → Next.js 프로젝트 data/ 폴더에 저장하세요")

In [ ]:
import random

with open("contents.json", "r", encoding="utf-8") as f:
    contents = json.load(f)

sample = random.choice(list(contents.values()))
print(f"제목: {sample['title']}")
print(f"장르: {sample['genre']} / 지역: {sample['region']}")
print(f"좌석: {sample['venue_seats']}석 / 인기도: {sample['popularity']}")
if sample.get('profile'):
    dp = sample['profile']['desire_profile']
    print(f"\n핵심 욕망: {dp['core_desire']}")
    print(f"감각 태그: {sample['profile']['sensory_tags']}")
    print(f"스펙트럼: {sample['profile']['spectrum_position']}/10")
print(f"\n임베딩 차원: {len(sample['embedding'])}")